In [2]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

In [3]:
# Ladataan datasetti
df_raw = pd.read_csv("cicids2017_cleaned.csv")
df100k = df_raw.sample(n=100000, random_state=42)
df = df100k[[
'Init_Win_bytes_forward',
'Flow IAT Mean',
'Packet Length Std',
'Subflow Fwd Bytes',
'Flow Duration',
'Bwd Packet Length Mean',
'Total Length of Fwd Packets',
'PSH Flag Count',
'Flow Packets/s',
'Destination Port',
'Attack Type'
]]

X = df.drop('Attack Type', axis=1)
y = df['Attack Type']

In [4]:
# Splittaus
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
   #('poly', PolynomialFeatures(degree=2, include_bias=False)), # TÄMÄ KOMMENTOITU ETTEI AJA KOKO PÄIVÄÄ
    ('logreg', LogisticRegression(class_weight='balanced', max_iter=1000)) # multi_class='multinomial' antoi herjan, että multi_class-parametri poistetaan käytöstä, joten sen jätin pois.
])

# Parametrit
param_grid = {
    'logreg__C': [0.1, 1, 10]
}

# GridSearchCV
grid_search = GridSearchCV(
    estimator = pipeline, 
    param_grid = param_grid,
    cv=5, 
    scoring='accuracy',
    verbose=1
)

# Etsitään parhaat parametrit
grid_search.fit(X_train_full, y_train_full)

# Tulokset
print(f"Parhaat parametrit: {grid_search.best_params_}")
print(f"_________________________________________________________")

# Lopullinen testi parhaalla mallilla
best_model = grid_search.best_estimator_

# TRAINING-RAPORTTI
train_preds = best_model.predict(X_train_full)
print("\n=== Training-raportti (Train Set) ===")
print(classification_report(y_train_full, train_preds))
print(confusion_matrix(y_train_full, train_preds))
print(f"_________________________________________________________")

# VALIDOINTIRAPORTTI
val_preds = cross_val_predict(best_model, X_train_full, y_train_full, cv=5)
print("\n=== Validointiraportti (Cross-Validation Predictions) ===")
print(classification_report(y_train_full, val_preds))
print(confusion_matrix(y_train_full, val_preds))
print(f"_________________________________________________________")

# LOPULLINEN TESTIRAPORTTI
test_preds = best_model.predict(X_test)
print("\n=== Lopullinen testi (Test Set): ===")
print(classification_report(y_test, test_preds))
print(confusion_matrix(y_test, test_preds))

Fitting 5 folds for each of 3 candidates, totalling 15 fits
Parhaat parametrit: {'logreg__C': 10}
_________________________________________________________

=== Training-raportti (Train Set) ===
                precision    recall  f1-score   support

          Bots       0.01      0.88      0.02        78
   Brute Force       0.12      1.00      0.21       310
          DDoS       0.50      1.00      0.66      4106
           DoS       0.68      0.85      0.75      6174
Normal Traffic       1.00      0.71      0.83     66378
 Port Scanning       0.69      0.97      0.80      2879
   Web Attacks       0.03      0.93      0.07        75

      accuracy                           0.74     80000
     macro avg       0.43      0.91      0.48     80000
  weighted avg       0.93      0.74      0.81     80000

[[   69     0     0     0     9     0     0]
 [    0   310     0     0     0     0     0]
 [    0    13  4090     1     1     1     0]
 [    0   229   415  5278   202     1    49]
 [ 806